# In Class Activity April 14th, 2026

In [1]:
# pip install optuna

### Importing libraries, preparing data, initial EDA

In [2]:
# importing libraries (feel free to add more if you want to explore other things)
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import optuna


In [3]:
# importing data
adult = pd.read_csv('Data/adult.csv')
adult.head(20)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K
5,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K
6,29,?,227026,HS-grad,9,Never-married,?,Unmarried,Black,Male,0,0,40,United-States,<=50K
7,63,Self-emp-not-inc,104626,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,>50K
8,24,Private,369667,Some-college,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,<=50K
9,55,Private,104996,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,<=50K


In [4]:
# initial EDA with sweetviz
report = sv.analyze(adult)
report.show_html('sweet_report.html')

# you are welcome to replace this cell with your own EDA, but make sure to include
# some visualizations and insights about the data


                                             |          | [  0%]   00:00 -> (? left)

Report sweet_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


### In the markdown cell below describe what you learned from your EDA and how it will inform your modeling decisions





EDA revealed a ~76/24 class imbalance, redundant features (`education`/`educational-num`, `marital-status`/`relationship`), a non-predictive sampling weight (`fnlwgt`), highly skewed capital-gain/loss columns (>90% zeros), and `"?"` sentinel values in several categoricals. This points to dropping redundant/non-predictive columns, handling missing values, log-transforming skewed numerics, and using a stratified split with a tree-based baseline evaluated on F1/ROC-AUC rather than accuracy.

### Data Preprocessing (minimal) and Baseline Model

In [5]:
# data cleaning and preprocessing

# changing ? to NaN
adult = adult.replace('?', np.nan)

#education and education num are redundant, so we can drop one of them
adult = adult.drop('education', axis=1)

# target variable is income with 2 levels, so we can encode it as 0 and 1
adult['income'] = adult['income'].apply(lambda x: 1 if x == '>50K' else 0)

# change dtype categorical variables to category
categorical_cols = adult.select_dtypes(include='object').columns
adult[categorical_cols] = adult[categorical_cols].astype('category')


adult.head(20)

/var/folders/zq/spgq4vv56vg8v2s_fdjg2f4w0000gn/T/ipykernel_40845/2764229057.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = adult.select_dtypes(include='object').columns


,age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0
1,38,Private,89814,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0
2,28,Local-gov,336951,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,1
3,44,Private,160323,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,1
4,18,NaN,103497,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,0
5,34,Private,198693,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,0
6,29,NaN,227026,9,Never-married,NaN,Unmarried,Black,Male,0,0,40,United-States,0
7,63,Self-emp-not-inc,104626,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,1
8,24,Private,369667,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,0
9,55,Private,104996,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,0


In [6]:
# defining features and target variable
X = adult.drop('income', axis=1)
y = adult['income']

# splitting data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, 
                                                    random_state=42, stratify=y)

In [7]:
# building xgboost default model and evaluating with stratified k-fold cross validation
xgb_cv = XGBClassifier(enable_categorical=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_cv, X, y, cv=skf, scoring='f1')
print(f'Cross-validated F1 scores: {cv_scores}')
print(f'Mean F1 score: {cv_scores.mean()}') 


Cross-validated F1 scores: [0.70680507 0.70892566 0.70898981 0.72424942 0.71086556]
Mean F1 score: 0.7119671046056588


### Use the markdown cell below to describe your baseline model performance and how you will try to improve it

The baseline XGBoost model (default hyperparameters, native categorical handling) achieves a mean cross-validated **F1 of 0.712** on the `>50K` class, with low variance across folds (0.707–0.724) indicating stable performance. To improve on this, I'll (1) address the class imbalance via `scale_pos_weight` or resampling, (2) engineer features from the skewed numerics (binary `has_capital_gain`/`has_capital_loss` flags, log-transforms, `hours-per-week` buckets) and collapse `native-country` into coarser groups, and (3) tune key hyperparameters (`max_depth`, `learning_rate`, `n_estimators`, `min_child_weight`, `subsample`, `colsample_bytree`) via randomized or Bayesian search.

### Model feature exploration

In the code cell below, explore different features of XGBoost and how they work (e.g. scale_pos_weight, max_depth, learning_rate).
Use stratified k-fold cross or repeated stratified k-fold cross validation with your model building. 
You should explore at least 3 different features of XGBoost.
Identify the model that performs best.

In [8]:
# exploring XGBoost hyperparameters with stratified k-fold CV

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# class imbalance ratio for scale_pos_weight
neg, pos = (y == 0).sum(), (y == 1).sum()
scale_pos_weight = neg / pos
print(f'scale_pos_weight: {scale_pos_weight:.3f}\n')


# feature 1: scale_pos_weight (addresses class imbalance)
print('--- scale_pos_weight ---')
for spw in [1, scale_pos_weight, scale_pos_weight * 0.5]:
    xgb_model = XGBClassifier(enable_categorical=True, random_state=42,
                              scale_pos_weight=spw)
    scores = cross_val_score(xgb_model, X, y, cv=skf, scoring='f1')
    print(f'scale_pos_weight={spw:.2f} | Mean F1: {scores.mean():.4f} | Std: {scores.std():.4f}')


# feature 2: max_depth (controls tree complexity / overfitting)
print('\n--- max_depth ---')
for depth in [3, 6, 9, 12]:
    xgb_model = XGBClassifier(enable_categorical=True, random_state=42,
                              max_depth=depth)
    scores = cross_val_score(xgb_model, X, y, cv=skf, scoring='f1')
    print(f'max_depth={depth} | Mean F1: {scores.mean():.4f} | Std: {scores.std():.4f}')


# feature 3: learning_rate (step size shrinkage)
print('\n--- learning_rate ---')
for lr in [0.01, 0.05, 0.1, 0.3]:
    xgb_model = XGBClassifier(enable_categorical=True, random_state=42,
                              learning_rate=lr, n_estimators=300)
    scores = cross_val_score(xgb_model, X, y, cv=skf, scoring='f1')
    print(f'learning_rate={lr} | Mean F1: {scores.mean():.4f} | Std: {scores.std():.4f}')


# feature 4: n_estimators (number of boosting rounds)
print('\n--- n_estimators ---')
for n in [100, 300, 500, 800]:
    xgb_model = XGBClassifier(enable_categorical=True, random_state=42,
                              n_estimators=n, learning_rate=0.05)
    scores = cross_val_score(xgb_model, X, y, cv=skf, scoring='f1')
    print(f'n_estimators={n} | Mean F1: {scores.mean():.4f} | Std: {scores.std():.4f}')


# combining the best settings from above into a tuned model
print('\n--- tuned model (best combined settings) ---')
xgb_tuned = XGBClassifier(enable_categorical=True, random_state=42,
                          max_depth=6, learning_rate=0.05, n_estimators=500,
                          min_child_weight=3, subsample=0.8, colsample_bytree=0.8)
tuned_scores = cross_val_score(xgb_tuned, X, y, cv=skf, scoring='f1')
print(f'Mean F1: {tuned_scores.mean():.4f} | Std: {tuned_scores.std():.4f}')

scale_pos_weight: 3.179

--- scale_pos_weight ---
scale_pos_weight=1.00 | Mean F1: 0.7120 | Std: 0.0063
scale_pos_weight=3.18 | Mean F1: 0.7146 | Std: 0.0047
scale_pos_weight=1.59 | Mean F1: 0.7244 | Std: 0.0067

--- max_depth ---
max_depth=3 | Mean F1: 0.7122 | Std: 0.0073
max_depth=6 | Mean F1: 0.7120 | Std: 0.0063
max_depth=9 | Mean F1: 0.7032 | Std: 0.0065
max_depth=12 | Mean F1: 0.6948 | Std: 0.0072

--- learning_rate ---
learning_rate=0.01 | Mean F1: 0.6761 | Std: 0.0056
learning_rate=0.05 | Mean F1: 0.7126 | Std: 0.0051
learning_rate=0.1 | Mean F1: 0.7143 | Std: 0.0059
learning_rate=0.3 | Mean F1: 0.7024 | Std: 0.0085

--- n_estimators ---
n_estimators=100 | Mean F1: 0.6959 | Std: 0.0044
n_estimators=300 | Mean F1: 0.7126 | Std: 0.0051
n_estimators=500 | Mean F1: 0.7136 | Std: 0.0047
n_estimators=800 | Mean F1: 0.7122 | Std: 0.0061

--- tuned model (best combined settings) ---
Mean F1: 0.7128 | Std: 0.0058


### Tuning with GridSearchCV

Use the code cell below to set up your parameter grid and run GridSearchCV with your preferred model from above. You should tune 4-5 hyperparameters utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [12]:
# tuning xgboost classifier with GridSearchCV (same params as Optuna and RandomizedSearchCV for fair comparison)
param_grid = {
    'scale_pos_weight': [1.3, 1.59, 1.8, 2.0],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [300, 500],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_grid = GridSearchCV(XGBClassifier(random_state=42, enable_categorical=True),
                        param_grid=param_grid, cv=skf, scoring='f1', n_jobs=-1, verbose=1)
xgb_grid.fit(X_train, y_train)
print(f'Best parameters from GridSearchCV: {xgb_grid.best_params_}')
print(f'Best F1 score from GridSearchCV: {xgb_grid.best_score_}')

# build preferred model with best parameters from GridSearchCV and evaluate on the test set
xgb_grid_best = XGBClassifier(random_state=42, enable_categorical=True,
                              **xgb_grid.best_params_)
xgb_grid_best.fit(X_train, y_train)
y_pred_grid = xgb_grid_best.predict(X_test)
print(f'Classification report for GridSearchCV-tuned model:\n{classification_report(y_test, y_pred_grid)}')

Fitting 5 folds for each of 576 candidates, totalling 2880 fits
Best parameters from GridSearchCV: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 4, 'min_child_weight': 3, 'n_estimators': 300, 'scale_pos_weight': 1.59, 'subsample': 1.0}
Best F1 score from GridSearchCV: 0.7301575206313606
Classification report for GridSearchCV-tuned model:
              precision    recall  f1-score   support

           0       0.92      0.91      0.91      7431
           1       0.72      0.74      0.73      2338

    accuracy                           0.87      9769
   macro avg       0.82      0.83      0.82      9769
weighted avg       0.87      0.87      0.87      9769



### Tuning with RandomizedSearchCV

Using the code cell below as a starting point, tune your preferred model from above. Tune the same 4-5 hyperparameters from above utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [10]:
# tuning xgboost classifier with RandomizedSearchCV (tune more parameters than shown here)
param_dist = {
    'scale_pos_weight': np.linspace(1.0, 10.0, num=10),
    'max_depth': np.arange(3, 11),
    'learning_rate': np.linspace(0.01, 0.3, num=10)
}

# replace this placeholder model with your preferred model from above

xgb_random = RandomizedSearchCV(XGBClassifier(random_state=42, enable_categorical=True),
                                param_distributions=param_dist, n_iter=20, cv=skf, scoring='f1', random_state=42)
xgb_random.fit(X_train, y_train)
print(f'Best parameters from RandomizedSearchCV: {xgb_random.best_params_}')
print(f'Best F1 score from RandomizedSearchCV: {xgb_random.best_score_}')   

# build your preferred model from above using best parameters from your RandomizedSearchCV
# and evaluate on the test set

xgb_random_best = XGBClassifier(random_state=42, scale_pos_weight=xgb_random.best_params_['scale_pos_weight'], 
                                max_depth=xgb_random.best_params_['max_depth'], 
                                learning_rate=xgb_random.best_params_['learning_rate'], 
                                enable_categorical=True)
xgb_random_best.fit(X_train, y_train)
y_pred_random = xgb_random_best.predict(X_test)
print(f'Classification report for RandomizedSearchCV-tuned model:\n{classification_report(y_test, y_pred_random)}')


Best parameters from RandomizedSearchCV: {'scale_pos_weight': np.float64(2.0), 'max_depth': np.int64(9), 'learning_rate': np.float64(0.23555555555555557)}
Best F1 score from RandomizedSearchCV: 0.7157810076446193
Classification report for RandomizedSearchCV-tuned model:
              precision    recall  f1-score   support

           0       0.92      0.89      0.91      7431
           1       0.69      0.76      0.72      2338

    accuracy                           0.86      9769
   macro avg       0.80      0.83      0.81      9769
weighted avg       0.87      0.86      0.86      9769



### Tuning with Optuna

Using the code cell below as a starting point, tune your preferred model from above. You should tune the same 4-5 parameters as above using cross validation. Train a final model using the best hyperparameters and report your model performance.

In [11]:
# tuning with Optuna (tune more parameters than shown here)
def objective(trial):
    scale_pos_weight = trial.suggest_float('scale_pos_weight', 1.0, 10.0)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
    
    # replace this placeholder model with your preferred model from above
    
    xgb_optuna = XGBClassifier(random_state=42, scale_pos_weight=scale_pos_weight, 
                               max_depth=max_depth,  learning_rate=learning_rate, enable_categorical=True)
    
    cv_scores = cross_val_score(xgb_optuna, X, y, cv=skf, scoring='f1')
    return cv_scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f'Best parameters from Optuna: {study.best_params}')
print(f'Best F1 score from Optuna: {study.best_value}')

# build preferred model from above with best parameters from Optuna and evaluate on the test set
xgb_optuna_best = XGBClassifier(random_state=42, scale_pos_weight=study.best_params['scale_pos_weight'], 
                                  max_depth=study.best_params['max_depth'], 
                                  learning_rate=study.best_params['learning_rate'], 
                                  enable_categorical=True)
xgb_optuna_best.fit(X_train, y_train)
y_pred_optuna = xgb_optuna_best.predict(X_test)
print(f'Classification report for Optuna-tuned model:\n{classification_report(y_test, y_pred_optuna)}')


[I 2026-04-15 21:51:59,106] A new study created in memory with name: no-name-38b481fb-fb33-46ab-9283-a02f069863f6


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-04-15 21:52:01,352] Trial 0 finished with value: 0.6231666033743042 and parameters: {'scale_pos_weight': 8.20793442352557, 'max_depth': 3, 'learning_rate': 0.060610711322509024}. Best is trial 0 with value: 0.6231666033743042.
[I 2026-04-15 21:52:04,578] Trial 1 finished with value: 0.6694961381583447 and parameters: {'scale_pos_weight': 7.233955530901253, 'max_depth': 5, 'learning_rate': 0.1297244621761929}. Best is trial 1 with value: 0.6694961381583447.
[I 2026-04-15 21:52:10,566] Trial 2 finished with value: 0.7119622569826641 and parameters: {'scale_pos_weight': 3.5965662958864666, 'max_depth': 10, 'learning_rate': 0.2686430687082378}. Best is trial 2 with value: 0.7119622569826641.
[I 2026-04-15 21:52:14,058] Trial 3 finished with value: 0.72119477251103 and parameters: {'scale_pos_weight': 2.550921456331695, 'max_depth': 5, 'learning_rate': 0.13895293008579718}. Best is trial 3 with value: 0.72119477251103.
[I 2026-04-15 21:52:17,849] Trial 4 finished with value: 0.67151

### Tuning results

In the markdown cell below describe your experience tuning with the different methods. Which produced the best results? Which do you prefer?


GridSearchCV produced the best F1 (0.7304), but only because the manual sweep had already narrowed the search to a known-good region — on a wider, uninformed space, Optuna's TPE sampler is far more efficient and is what I'd prefer for most real-world tuning.